# Agentic AI, RAG & LLMs — Hands-On WorkshopWelcome! In the next two hours you will:1. Call a large language model from Python2. Learn how prompting changes what you get back3. Build a chatbot that remembers the conversation4. Build a **RAG** system from scratch (retrieval over your own documents)5. Build an **agent** that decides which Python function to call**You do not need to know Python well.** Every cell either runs as-is or has aclearly marked `TODO` for you to fill in.**How to run a cell:** click it, then press `Shift + Enter`.

---## Part 0 — Setup check (10 min)Run the next two cells. If you see `Setup OK`, you are ready.If a "Select Kernel" prompt appears at the top, choose **Python Environments**and then the **3.11** entry.

In [ ]:
# Installs everything we need. Usually instant — it's pre-installed for you.%pip install -q -U google-genai sentence-transformers numpyprint("Install finished.")

If you see a note saying *"you may need to restart the kernel"* — ignore it.Everything is already installed. Do **not** click Restart.

In [ ]:
from google import genaifrom google.genai import typesimport numpy as npimport logging# Hides a harmless SDK warning about "automatic function calling".logging.getLogger("google_genai.models").setLevel(logging.ERROR)MODEL = "gemini-2.5-flash"            # our main model for the whole workshopEMBED_MODEL = "gemini-embedding-001"  # a backup for the RAG sectionprint("Setup OK")

### Your API keyPaste the Gemini key you created at [aistudio.google.com](https://aistudio.google.com)into the cell below, between the quotes.> **Keep your own key.** Everyone must use their own — the free tier counts requests> per account, so sharing one key will rate-limit the whole room.

In [ ]:
API_KEY = "PASTE_YOUR_GEMINI_KEY_HERE"client = genai.Client(api_key=API_KEY)print("Client ready.")

---## Part 1 — Your first LLM call (10 min)An LLM takes text in and produces text out. That is the whole interface.- **Prompt** = the text you send- **Token** = roughly 3-4 characters; models read and write in tokens- **Model** = the specific trained system answering you (`gemini-2.5-flash` here)

In [ ]:
response = client.models.generate_content(    model=MODEL,    contents="Explain what a large language model is, in one sentence.")print(response.text)

In [ ]:
# TODO: change the question below to anything you're curious about, then run the cell.my_question = "TODO: write your own question here"response = client.models.generate_content(model=MODEL, contents=my_question)print(response.text)

**Notice:** run the same prompt twice and you may get different wording. LLMs areprobabilistic, not lookup tables. We control that next.

---## Part 2 — Prompting basics (20 min)Three levers you will use constantly:| Lever | What it does ||---|---|| **System instruction** | Standing orders — persona, rules, format || **Temperature** | 0.0 = focused and repeatable, 2.0 = wild and creative || **Few-shot examples** | Show 2-3 examples of the output you want |

In [ ]:
# A system instruction shapes every answer without cluttering your question.response = client.models.generate_content(    model=MODEL,    contents="What is a database?",    config=types.GenerateContentConfig(        system_instruction="You explain things to a curious 10-year-old. Two sentences maximum.",        temperature=0.2,    ),)print(response.text)

In [ ]:
# TODO: write a system instruction that makes the model answer as a pirate,# and set temperature to 1.5 to make it more playful.response = client.models.generate_content(    model=MODEL,    contents="What is a database?",    config=types.GenerateContentConfig(        system_instruction="TODO",        temperature=0.0,  # TODO: change this    ),)print(response.text)

### Structured outputFree-form text is hard for programs to use. Ask for JSON and you can parse it.

In [ ]:
import jsonresponse = client.models.generate_content(    model=MODEL,    contents="List 3 fruits with their colours.",    config=types.GenerateContentConfig(        system_instruction="Reply with valid JSON only. No markdown, no backticks.",        temperature=0.0,    ),)data = json.loads(response.text)print(data)print(type(data))

**Look closely at that output.** You probably got something like`{'fruits': [...]}` — an *object wrapping* the list, not a bare list.This is the single most common surprise when working with JSON from an LLM:"reply with JSON" is not specific enough. If your code expects a list and gets adictionary, it breaks. **Describe the exact shape you want**, as below.

In [ ]:
# TODO: ask for 4 European cities, each with its country and population.# Be specific about the shape: a JSON ARRAY, with keys "city", "country", "population".# Then print just the city names.response = client.models.generate_content(    model=MODEL,    contents="TODO: your request here",    config=types.GenerateContentConfig(        system_instruction="Reply with valid JSON only. No markdown, no backticks.",        temperature=0.0,    ),)data = json.loads(response.text)# TODO: print the city names

---## Part 3 — Multi-turn chat (15 min)`generate_content` has **no memory** — each call starts from nothing.A chat session keeps the history and resends it every turn. That is all "memory" is.

In [ ]:
# Proof that a single call has no memory:print(client.models.generate_content(model=MODEL, contents="My name is Sam.").text)print("---")print(client.models.generate_content(model=MODEL, contents="What is my name?").text)

In [ ]:
# Now with a chat session, which carries the history.# The system instruction keeps answers short so we can see all the turns at once.chat = client.chats.create(    model=MODEL,    config=types.GenerateContentConfig(        system_instruction="Keep every answer to 3 sentences maximum."    ),)print(chat.send_message("My name is Sam and I have 2 dogs.").text)print("---")print(chat.send_message("How many paws is that, and what is my name?").text)

In [ ]:
# TODO: have a 3-turn conversation of your own with the model.chat = client.chats.create(    model=MODEL,    config=types.GenerateContentConfig(        system_instruction="Keep every answer to 3 sentences maximum."    ),)print(chat.send_message("TODO: turn 1").text)# TODO: turn 2# TODO: turn 3# This shows what actually gets resent to the model every turn:for message in chat.get_history():    print(message.role, "->", message.parts[0].text[:80])

The third answer knows you said April. Nothing magic happened — the wholeconversation was resent with every message. That is why long chats cost more andeventually hit a limit.

---## Part 4 — RAG from scratch (35 min)**The problem:** the model does not know your company handbook, your notes, oranything written after its training cut-off. Asked anyway, it may invent an answer.**RAG** (Retrieval-Augmented Generation) fixes this in three steps:1. **Embed** — turn each document into a list of numbers that captures its meaning2. **Retrieve** — find the documents whose numbers are closest to the question's numbers3. **Generate** — paste those documents into the prompt and ask the modelWe are building this ourselves, with numpy. No vector database, no framework.

### Step 1 — EmbeddingsAn embedding turns text into a vector (a list of numbers). Similar meanings landclose together.The next cell sets up our embedding function. It uses a small model that runslocally, and automatically falls back to the Gemini embedding API if that modelcan't be downloaded.

In [ ]:
try:    from sentence_transformers import SentenceTransformer    _local_model = SentenceTransformer("all-MiniLM-L6-v2")    def embed_texts(texts):        """Turn a list of strings into normalised vectors."""        return _local_model.encode(texts, normalize_embeddings=True)    print("Embedding backend: local model (all-MiniLM-L6-v2)")except Exception as error:    print("Local embedding model unavailable —", type(error).__name__)    print("Falling back to the Gemini embedding API. Everything below still works.")    def embed_texts(texts):        """Turn a list of strings into normalised vectors, via the Gemini API."""        result = client.models.embed_content(model=EMBED_MODEL, contents=list(texts))        vectors = np.array([e.values for e in result.embeddings], dtype=float)        return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)    print("Embedding backend: Gemini API (gemini-embedding-001)")

In [ ]:
vector = embed_texts(["The cat sat on the mat."])[0]print("Length of the vector:", len(vector))print("First 8 numbers:", vector[:8])

In [ ]:
# Similar meanings -> high similarity. Different meanings -> low.def similarity(a, b):    va, vb = embed_texts([a, b])    return float(va @ vb)   # dot product of normalised vectors = cosine similarityprint("dog/puppy      ", round(similarity("I love dogs", "Puppies are wonderful"), 3))print("dog/tax return ", round(similarity("I love dogs", "Please file your tax return"), 3))

### Step 2 — RetrievalHere is our tiny "knowledge base". In a real system these would be chunks of yourown PDFs or wiki pages.

In [ ]:
documents = [    "The Eiffel Tower is in Paris and was completed in 1889.",    "Python is a programming language created by Guido van Rossum in 1991.",    "The Great Wall of China is over 21,000 kilometres long.",    "Our office wifi password is 'workshop2026' and the guest network is 'GuestNet'.",    "The company holiday policy allows 25 days of paid leave per year.",    "Coffee is made from roasted coffee beans grown near the equator.",]# Embed every document once, up front.doc_vectors = embed_texts(documents)print("Shape:", doc_vectors.shape, "-> one row per document")

In [ ]:
def retrieve(question, k=2):    """Return the k documents most similar in meaning to the question."""    q_vector = embed_texts([question])[0]    # TODO: compute the similarity of the question to every document.    # Hint: doc_vectors @ q_vector    scores = None    top_indexes = np.argsort(scores)[::-1][:k]    return [documents[i] for i in top_indexes]print(retrieve("What is the wifi password?"))

### How good are those matches, really?The next cell shows the similarity **score** alongside each result. This mattersmore than it looks.

In [ ]:
def retrieve_with_scores(question, k=2):    q_vector = embed_texts([question])[0]    scores = doc_vectors @ q_vector    top_indexes = np.argsort(scores)[::-1][:k]    return [(round(float(scores[i]), 3), documents[i]) for i in top_indexes]for question in ["What is the wifi password?",                 "how much time off do I get?",                 "tell me about tall buildings in France"]:    print(question)    for score, doc in retrieve_with_scores(question):        print("   ", score, "|", doc)    print()

**Two things to notice.**First, "how much time off do I get?" finds the holiday policy even though itshares almost no words with it. Embeddings match **meaning**, not keywords.Second, look at the second result for each question — often the same irrelevantdocument, with a much lower score. **Retrieval always returns `k` documents,whether or not any of them are any good.** It has no idea what "relevant" means;it just ranks. Guarding against that is the next step's job.

### Step 3 — GenerateFirst, see the model fail without retrieval. Then give it the context.

In [ ]:
question = "What is the office wifi password?"# WITHOUT retrieval — the model has no way to know this.print(client.models.generate_content(model=MODEL, contents=question).text)

In [ ]:
def ask_with_rag(question, k=2):    # TODO: 1. retrieve the relevant documents and join them into one string    context = None    # TODO: 2. build a prompt that includes the context and the question    prompt = None    response = client.models.generate_content(        model=MODEL,        contents=prompt,        config=types.GenerateContentConfig(            system_instruction=(                "Answer using ONLY the provided context. "                "If the answer is not in the context, say you don't know."            ),            temperature=0.0,        ),    )    return response.textprint(ask_with_rag("What is the office wifi password?"))

In [ ]:
# The important test: does it admit when it doesn't know?print(ask_with_rag("How many days of leave do I get?"))print("---")print(ask_with_rag("Who won the 2026 World Cup?"))

The second answer is the one that matters. Retrieval still handed the model twodocuments — they were just irrelevant, exactly as the scores predicted. The"answer only from the context" instruction is what turned a probablehallucination into an honest "I don't know".

In [ ]:
# TODO: add 2-3 facts about YOURSELF or your team to `documents`,# re-embed, then ask a question only your new facts can answer.documents.append("TODO: your fact here")# TODO: re-run the embedding step so doc_vectors includes your new documentsdoc_vectors = Noneprint(ask_with_rag("TODO: your question here"))

**That is RAG.** Roughly 15 lines. Production systems add chunking, betterembedding models, a vector database (Chroma, FAISS, Qdrant) and re-ranking — but thethree steps never change.

---## Part 5 — Your first agent (25 min)So far the model only produces text. An **agent** can *act*: you hand it somePython functions ("tools"), and it decides on its own which to call, with whicharguments, and what to do with the result.The loop is: **think → choose a tool → run it → read the result → answer.**Two things make a good tool: a clear **docstring** and **type hints**. The SDK readsboth to tell the model what the tool does.

In [ ]:
# LLMs are famously unreliable at arithmetic. Watch:print(client.models.generate_content(    model=MODEL,    contents="What is 4738 * 2913? Reply with just the number.",).text)print("The real answer is:", 4738 * 2913)

### Read those two numbers carefullyCompare them digit by digit. They are probably very close — differing somewhere inthe middle — rather than wildly different.This is the most important thing to understand about LLM failure. The model is notcalculating and getting it wrong; it is **predicting what a plausible answer lookslike**, and a plausible product of two 4-digit numbers is an 8-digit number thatstarts and ends about right.Wrong answers rarely look wrong. That is exactly why we give it a tool.

In [ ]:
def multiply(a: float, b: float) -> float:    """Multiply two numbers together and return the result."""    print(f"   [tool called: multiply({a}, {b})]")    return a * bresponse = client.models.generate_content(    model=MODEL,    contents="What is 4738 * 2913?",    config=types.GenerateContentConfig(tools=[multiply]),)print(response.text)

You should see the `[tool called: ...]` line print, and the answer is now exact.The model did not do the maths — it recognised it needed a tool, called your Pythonfunction, and used what came back.Now let's give it more than one tool, including our RAG retriever.

In [ ]:
def lookup_knowledge_base(query: str) -> str:    """Look up facts about the office, company policy and general trivia."""    print(f"   [tool called: lookup_knowledge_base('{query}')]")    return " ".join(retrieve(query))def word_count(text: str) -> int:    """Count how many words are in a piece of text."""    print(f"   [tool called: word_count(...)]")    return len(text.split())# TODO: pass all three tools (multiply, lookup_knowledge_base, word_count)# and ask a question that needs TWO of them.response = client.models.generate_content(    model=MODEL,    contents="TODO: your question here",    config=types.GenerateContentConfig(tools=[]),  # TODO)print(response.text)

Watch which tools printed. The model picked them itself, in the right order, andcombined the results — you never wrote an `if` statement deciding when to searchversus when to calculate. **That is the agentic part.**

In [ ]:
# TODO: write your own tool and give it to the model.# Ideas: get today's date, convert Celsius to Fahrenheit, reverse a string,# check if a number is prime.def my_tool(TODO: str) -> str:    """TODO: describe clearly what this does — the model reads this docstring."""    print("   [tool called: my_tool]")    return "TODO"response = client.models.generate_content(    model=MODEL,    contents="TODO: a question that needs your tool",    config=types.GenerateContentConfig(tools=[my_tool]),)print(response.text)

---## Part 6 — Where to go next (10 min)You have now built, from scratch, the three things every LLM product is made of.**What you'd add for production**- *Chunking* — split long documents into ~500-token pieces before embedding- *A vector database* — `chromadb` is the easiest next step; FAISS or Qdrant at scale- *Better embeddings* — BGE, GTE or `gemini-embedding-001` beat MiniLM on accuracy- *Evaluation* — measure whether retrieved documents were actually relevant- *Guardrails* — the "answer only from context" instruction is your first defence  against hallucination, not your last**Frameworks worth learning next**| Tool | Good for ||---|---|| **LangChain / LangGraph** | Wiring multi-step pipelines and stateful agents || **LlamaIndex** | RAG-focused: loaders, chunking, indexes || **smolagents** (Hugging Face) | Minimal agents that write and run code || **PydanticAI** | Type-safe agents with very little boilerplate |**Free courses**- Hugging Face Agents Course — `huggingface.co/learn/agents-course` (free, Apache-2.0)- Google AI Studio docs — `ai.google.dev/gemini-api/docs`**One safety note:** free-tier prompts may be used to improve the provider's models.Never paste customer data, credentials or anything confidential into a free-tier API.---### Appendix — Backup provider (only if Gemini is rate-limited)If you hit a `429 RESOURCE_EXHAUSTED` error, switch to Groq. Same lesson, one changed cell.

In [ ]:
# Only run this if Gemini is unavailable.# %pip install -q openai## from openai import OpenAI# groq = OpenAI(api_key="YOUR_GROQ_KEY", base_url="https://api.groq.com/openai/v1")# reply = groq.chat.completions.create(#     model="llama-3.3-70b-versatile",#     messages=[{"role": "user", "content": "Hello!"}],# )# print(reply.choices[0].message.content)